# Test / demo — Open Transit Delay pipeline

Runs the pipeline end-to-end on the **shipped pseudonymized sample** (one Umlauf/day):
matches the vehicle, applies the confidence gate, reconstructs arrivals and computes
the per-stop delay. A quick way for users and reviewers to confirm the scripts work.


In [ ]:
import os
import sys
from pathlib import Path

# Locate the repository even when the notebook kernel starts elsewhere.
ROOT = Path.cwd().resolve()
while not (ROOT / 'pyproject.toml').exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
os.chdir(ROOT)
sys.path.insert(0, str(ROOT / 'src'))
print('cwd:', os.getcwd())

import pandas as pd
from itsc_delay_pipeline.config import load_config
from itsc_delay_pipeline.pipeline import run_pipeline

cfg = load_config(ROOT / 'config.yaml')
cfg.raw.setdefault('run', {})['tag'] = cfg.raw['run'].get('tag', '2026-01-08')
umlauf, tag = str(cfg.raw['run']['umlauf_id']), cfg.tag
print('umlauf', umlauf, '| tag', tag)

## 1. Run the pipeline (match vehicle + reconstruct arrivals + delays)

In [ ]:
out = run_pipeline(cfg)
out

## 2. Matched vehicle + confidence gate

In [ ]:
va = pd.read_csv(out['vehicle_assignment_csv'])
row = va.iloc[0]
print(f"vehicle {row.vehicle_id} | confident={int(row.confident)} | mrel={row.mrel:.3f} "
      f"(top {row.top_hits:.0f} / second {row.second_hits:.0f}, {row.n_fahrten} Fahrten)")
assert row.confident == 1, 'expected a confident vehicle match on the sample'
print('OK: vehicle matched, confident')

## 3. Per-stop delays (delay_s = actual - scheduled)

In [ ]:
arr = pd.read_csv(out['csv'], low_memory=False)
delays = arr['delay_s'].dropna()
print(f'{len(delays)} stops with a delay | median {delays.median():.0f} s | mean {delays.mean():.0f} s')
assert len(delays) > 100, 'expected many stops with a computed delay'
arr[['frt_fid','stop_seq','ort_name','ankunft_soll','ankunft_ist','delay_s']].dropna(subset=['delay_s']).head(10)

## 4. Delay along one Fahrt

In [ ]:
import matplotlib.pyplot as plt
frt = arr.dropna(subset=['delay_s'])['frt_fid'].value_counts().index[0]
s = arr[arr['frt_fid'] == frt].sort_values('stop_seq')
plt.figure(figsize=(8,3))
plt.plot(s['stop_seq'], s['delay_s'], marker='o')
plt.axhline(0, color='grey', lw=.8); plt.xlabel('stop_seq'); plt.ylabel('delay_s'); plt.title(f'Fahrt {frt}')
plt.tight_layout(); plt.show()

## 5. (optional) delay-construction inspection figure
`itsc-delay viz-delay -c config.yaml` renders a small-subsample teaching figure.

In [ ]:
from itsc_delay_pipeline.viz_delay import viz_delay
png = viz_delay(cfg, n_stops=6)
print('wrote', png)